In [6]:
import pandas as pd
import numpy as np

# 1. 두 개의 CSV 파일 로드
try:
    df_main = pd.read_csv("data.csv")
    df_updated = pd.read_csv("data_updated.csv")
    print("✅ 'data.csv'와 'data_updated.csv' 파일을 성공적으로 불러왔습니다.")
except FileNotFoundError as e:
    print(f"❌ 파일 로드 실패: {e}. 업로드한 파일 이름을 확인해주세요.")
    # Stop execution if files aren't found
    exit()
except Exception as e:
    print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
    exit()


# --- [!!! 핵심 수정 !!!] ---
# '시즌' 컬럼의 형식을 통일합니다.
# 두 파일 모두 '시즌' 컬럼이 있는지 확인
if '시즌' not in df_main.columns or '시즌' not in df_updated.columns:
    print("❌ '시즌' 컬럼을 찾을 수 없습니다. 컬럼명을 확인해주세요.")
    exit()

# data.csv의 '시즌' (예: '2016/2017')을 '2016-2017' 형식으로 변경
df_main['시즌'] = df_main['시즌'].astype(str).str.replace('/', '-')
# data_updated.csv도 만약을 위해 형식을 통일
df_updated['시즌'] = df_updated['시즌'].astype(str).str.replace('/', '-')
print("✅ '시즌' 컬럼 형식을 'YYYY-YYYY'로 통일했습니다.")
# --- [수정 완료] ---


# 2. data_updated.csv에서 기준이 될 정보(truth)만 추출
truth_columns = ['선수명', '시즌', '포지션', '시즌종료시점만나이']
if not all(col in df_updated.columns for col in truth_columns):
    missing_cols = [col for col in truth_columns if col not in df_updated.columns]
    print(f"❌ 'data_updated.csv' 파일에 필요한 컬럼이 누락되었습니다: {missing_cols}")
    exit()

df_truth = df_updated[truth_columns].copy()
df_truth = df_truth.drop_duplicates(subset=['선수명', '시즌'], keep='first')

# 3. 효율적인 수정을 위해 두 데이터프레임의 인덱스를 (선수명, 시즌)으로 설정
try:
    if df_main.duplicated(subset=['선수명', '시즌']).any():
        print("⚠️ 'data.csv'의 ('선수명', '시즌') 키에 중복이 발견되어, 첫 번째 값 기준으로 처리합니다.")
        df_main = df_main.drop_duplicates(subset=['선수명', '시즌'], keep='first')
        
    # 원본 'data.csv'의 컬럼 순서를 저장해 둠
    original_columns = df_main.columns.tolist()
        
    df_main = df_main.set_index(['선수명', '시즌'])
    df_truth = df_truth.set_index(['선수명', '시즌'])
except KeyError:
    print("❌ 'data.csv' 또는 'data_updated.csv'에 '선수명' 또는 '시즌' 컬럼이 없습니다.")
    exit()

# 4. 요청하신 작업 수행

# 작업 1: '포지션' 업데이트 (data_updated 기준으로 덮어쓰기)
print("\n[작업 1] '포지션' 컬럼을 'data_updated.csv' 기준으로 업데이트합니다...")
original_positions = df_main['포지션'].copy()
df_main['포지션'].update(df_truth['포지션'])
position_changes = (df_main['포지션'] != original_positions).sum()
print(f"  -> 총 {position_changes}개의 '포지션' 정보가 업데이트되었습니다.")

# 작업 2: '시즌종료시점만나이'의 비어있는(NaN) 값 채우기
print("\n[작업 2] '시즌종료시점만나이'의 비어있는(NaN) 값을 'data_updated.csv' 기준으로 채웁니다...")
original_na_count = df_main['시즌종료시점만나이'].isna().sum()
df_main['시즌종료시점만나이'] = df_main['시즌종료시점만나이'].fillna(df_truth['시즌종료시점만나이'])
filled_na_count = original_na_count - df_main['시즌종료시점만나이'].isna().sum()
print(f"  -> 총 {filled_na_count}개의 '시즌종료시점만나이' 정보가 채워졌습니다.")

# 5. 인덱스 리셋 및 원본 컬럼 순서 복원 후 저장
df_corrected = df_main.reset_index()

# 원본 'data.csv'의 컬럼 순서를 그대로 유지하기
# (업데이트 과정에서 순서가 바뀌었을 수 있으므로)
final_columns = [col for col in original_columns if col in df_corrected.columns]
# 혹시 모를 누락/추가 컬럼에 대비
final_columns += [col for col in df_corrected.columns if col not in final_columns]
df_corrected = df_corrected[final_columns]

output_filename = 'data_corrected.csv'
df_corrected.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"\n✅ 모든 작업이 완료되었습니다. '{output_filename}' 파일로 저장되었습니다.")

✅ 'data.csv'와 'data_updated.csv' 파일을 성공적으로 불러왔습니다.
✅ '시즌' 컬럼 형식을 'YYYY-YYYY'로 통일했습니다.
⚠️ 'data.csv'의 ('선수명', '시즌') 키에 중복이 발견되어, 첫 번째 값 기준으로 처리합니다.

[작업 1] '포지션' 컬럼을 'data_updated.csv' 기준으로 업데이트합니다...
  -> 총 913개의 '포지션' 정보가 업데이트되었습니다.

[작업 2] '시즌종료시점만나이'의 비어있는(NaN) 값을 'data_updated.csv' 기준으로 채웁니다...
  -> 총 0개의 '시즌종료시점만나이' 정보가 채워졌습니다.

✅ 모든 작업이 완료되었습니다. 'data_corrected.csv' 파일로 저장되었습니다.


In [7]:
import pandas as pd
import numpy as np

print("데이터 업데이트 스크립트를 시작합니다...")

# --- 1. 업데이트할 정보 (제공해주신 목록) ---
# 제공해주신 214개 항목을 리스트로 변환
update_data_list = [
    ('2016/2017', 'raffael', '중앙 공격수', 32),
    ('2017/2018', 'raffael', '중앙 공격수', 33),
    ('2017/2018', 'ihlas bebou', '오른쪽 미드필더', 24),
    ('2017/2018', 'theodor gebre selassie', '오른쪽 수비수', 31),
    ('2017/2018', 'philipp bargfrede', '중앙 수비형 미드필더', 29),
    ('2017/2018', 'martin harnik', '중앙 공격수', 31),
    ('2017/2018', 'patrick herrmann', '오른쪽 미드필더', 27),
    ('2017/2018', 'matthias lehmann', '중앙 수비형 미드필더', 35),
    ('2017/2018', 'lukas rupp', '중앙 미드필더', 27),
    ('2017/2018', 'robin quaison', '중앙 공격수', 24),
    ('2018/2019', 'wout weghorst', '중앙 공격수', 26),
    ('2018/2019', 'salif sané', '중앙 수비수', 28),
    ('2018/2019', 'theodor gebre selassie', '오른쪽 수비수', 32),
    ('2018/2019', 'adam bodzek', '중앙 수비형 미드필더', 33),
    ('2018/2019', 'patrick herrmann', '오른쪽 미드필더', 28),
    ('2018/2019', 'martin harnik', '중앙 공격수', 32),
    ('2018/2019', 'hanno behrens', '중앙 미드필더', 29),
    ('2018/2019', 'javairô dilrosun', '왼쪽 윙어', 21),
    ('2018/2019', 'claudio pizarro', '중앙 공격수', 40),
    ('2018/2019', 'robin quaison', '중앙 공격수', 25),
    ('2018/2019', 'florent muslija', '공격형 미드필더', 20),
    ('2018/2019', 'virgil misidjan', '오른쪽 윙어', 25),
    ('2019/2020', 'wout weghorst', '중앙 공격수', 27),
    ('2019/2020', 'patrick herrmann', '오른쪽 미드필더', 29),
    ('2019/2020', 'christopher lenz', '왼쪽 수비수', 25),
    ('2019/2020', 'robin quaison', '중앙 공격수', 26),
    ('2019/2020', 'javairô dilrosun', '왼쪽 윙어', 22),
    ('2019/2020', 'theodor gebre selassie', '오른쪽 수비수', 33),
    ('2019/2020', 'ihlas bebou', '오른쪽 미드필더', 26),
    ('2019/2020', 'marius wolf', '오른쪽 미드필더', 25),
    ('2019/2020', 'adam bodzek', '중앙 수비형 미드필더', 34),
    ('2020/2021', 'wout weghorst', '중앙 공격수', 28),
    ('2020/2021', 'silas katompa mvumpa', '오른쪽 미드필더', 22),
    ('2020/2021', 'christopher lenz', '왼쪽 수비수', 26),
    ('2020/2021', 'ihlas bebou', '오른쪽 미드필더', 27),
    ('2020/2021', 'aymen barkok', '공격형 미드필더', 23),
    ('2020/2021', 'marius wolf', '오른쪽 미드필더', 26),
    ('2020/2021', 'theodor gebre selassie', '오른쪽 수비수', 34),
    ('2020/2021', 'robin quaison', '중앙 공격수', 27),
    ('2020/2021', "mu'nas dabbur", '중앙 공격수', 29),
    ('2020/2021', 'patrick herrmann', '오른쪽 미드필더', 30),
    ('2021/2022', 'wout weghorst', '중앙 공격수', 29),
    ('2021/2022', 'ihlas bebou', '중앙 공격수', 28),
    ('2021/2022', 'marius wolf', '오른쪽 미드필더', 27),
    ('2021/2022', 'patrick herrmann', '오른쪽 미드필더', 31),
    ('2021/2022', "mu'nas dabbur", '중앙 공격수', 30),
    ('2022/2023', 'marius wolf', '오른쪽 미드필더', 28),
    ('2022/2023', 'ihlas bebou', '중앙 공격수', 29),
    ('2022/2023', 'christopher lenz', '왼쪽 수비수', 28),
    ('2022/2023', "mu'nas dabbur", '중앙 공격수', 31),
    ('2022/2023', 'silas katompa mvumpa', '오른쪽 미드필더', 24),
    ('2022/2023', 'aymen barkok', '공격형 미드필더', 25),
    ('2023/2024', 'wout weghorst', '중앙 공격수', 31),
    ('2023/2024', 'ihlas bebou', '중앙 공격수', 30),
    ('2023/2024', 'marius wolf', '오른쪽 수비수', 29),
    ('2023/2024', 'silas katompa mvumpa', '오른쪽 미드필더', 25),
    ('2024/2025', 'mohammed amoura', '중앙 공격수', 25),
    ('2024/2025', 'frank onyeka', '중앙 미드필더', 27),
    ('2016/2017', 'steven nzonzi', '중앙 수비형 미드필더', 28),
    ('2016/2017', 'raphaël varane', '중앙 수비수', 24),
    ('2016/2017', 'pedro léon', '오른쪽 미드필더', 30),
    ('2016/2017', 'antonio luna', '왼쪽 수비수', 26),
    ('2016/2017', 'emre colak', '공격형 미드필더', 26),
    ('2016/2017', 'roberto torres', '공격형 미드필더', 28),
    ('2016/2017', 'raul navas', '중앙 수비수', 29),
    ('2016/2017', 'ibai gomez', '왼쪽 미드필더', 27),
    ('2016/2017', 'martin mantovani', '중앙 수비수', 32),
    ('2016/2017', 'alejandro arribas', '중앙 수비수', 28),
    ('2016/2017', 'gaizka toquero', '중앙 공격수', 32),
    ('2016/2017', 'isma lopez', '왼쪽 수비수', 27),
    ('2017/2018', 'steven nzonzi', '중앙 수비형 미드필더', 29),
    ('2017/2018', 'lucas pérez', '중앙 공격수', 29),
    ('2017/2018', 'raphaël varane', '중앙 수비수', 25),
    ('2017/2018', 'raul navas', '중앙 수비수', 30),
    ('2017/2018', 'ibai gomez', '왼쪽 미드필더', 28),
    ('2017/2018', 'antonio luna', '왼쪽 수비수', 27),
    ('2017/2018', 'emre colak', '공격형 미드필더', 27),
    ('2017/2018', 'javi castellano', '중앙 수비형 미드필더', 30),
    ('2017/2018', 'omar ramos', '왼쪽 미드필더', 30),
    ('2017/2018', 'diego rolan', '중앙 공격수', 25),
    ('2018/2019', 'raphaël varane', '중앙 수비수', 26),
    ('2018/2019', 'giannelli imbula', '중앙 미드필더', 26),
    ('2018/2019', 'michel herrero', '중앙 미드필더', 30),
    ('2018/2019', 'ibai gomez', '왼쪽 미드필더', 29),
    ('2018/2019', 'néstor araújo', '중앙 수비수', 27),
    ('2018/2019', 'javi moyano', '오른쪽 수비수', 33),
    ('2018/2019', 'david ferreiro', '오른쪽 미드필더', 31),
    ('2019/2020', 'roberto torres', '공격형 미드필더', 31),
    ('2019/2020', 'raphaël varane', '중앙 수비수', 27),
    ('2019/2020', 'lucas pérez', '중앙 공격수', 31),
    ('2019/2020', 'javi moyano', '오른쪽 수비수', 34),
    ('2019/2020', 'michel herrero', '중앙 미드필더', 31),
    ('2019/2020', 'néstor araújo', '중앙 수비수', 28),
    ('2019/2020', 'pedro léon', '오른쪽 미드필더', 33),
    ('2020/2021', 'raphaël varane', '중앙 수비수', 28),
    ('2020/2021', 'lucas pérez', '중앙 공격수', 32),
    ('2020/2021', 'david ferreiro', '오른쪽 미드필더', 33),
    ('2020/2021', 'roberto torres', '공격형 미드필더', 32),
    ('2020/2021', 'néstor araújo', '중앙 수비수', 29),
    ('2020/2021', 'michel herrero', '중앙 미드필더', 32),
    ('2021/2022', 'lucas pérez', '중앙 공격수', 33),
    ('2021/2022', 'néstor araújo', '중앙 수비수', 30),
    ('2021/2022', 'roberto torres', '공격형 미드필더', 33),
    ('2022/2023', 'inigo eguaras', '중앙 수비형 미드필더', 31),
    ('2023/2024', 'josé maria callejon', '오른쪽 윙어', 37),
    ('2024/2025', 'marash kumbulla', '중앙 수비수', 25),
    ('2016/2017', 'dalbert', '왼쪽 수비수', 23),
    ('2016/2017', 'paul baysse', '중앙 수비수', 29),
    ('2016/2017', 'younousse sankharé', '중앙 미드필더', 27),
    ('2016/2017', 'yoann gourcuff', '공격형 미드필더', 30),
    ('2016/2017', 'stephane sessegnon', '공격형 미드필더', 33),
    ('2016/2017', 'tobias badila', '왼쪽 수비수', 24),
    ('2016/2017', 'igor lewczuk', '중앙 수비수', 31),
    ('2016/2017', 'jordan marie', '중앙 미드필더', 25),
    ('2016/2017', 'diego rolan', '중앙 공격수', 24),
    ('2016/2017', 'somalia', '중앙 수비형 미드필더', 28),
    ('2017/2018', 'giannelli imbula', '중앙 미드필더', 25),
    ('2017/2018', 'adama mbengue', '왼쪽 수비수', 24),
    ('2017/2018', 'younousse sankharé', '중앙 미드필더', 28),
    ('2017/2018', 'somalia', '중앙 수비형 미드필더', 29),
    ('2017/2018', 'jordan marie', '중앙 미드필더', 26),
    ('2017/2018', 'stéphane darbion', '왼쪽 미드필더', 34),
    ('2017/2018', 'stef peeters', '중앙 미드필더', 26),
    ('2018/2019', 'ghislain konan', '왼쪽 수비수', 23),
    ('2018/2019', 'jordan marie', '중앙 미드필더', 27),
    ('2018/2019', 'anthony briancon', '중앙 수비수', 24),
    ('2018/2019', 'younousse sankharé', '중앙 미드필더', 29),
    ('2018/2019', 'paul baysse', '중앙 수비수', 31),
    ('2019/2020', 'ghislain konan', '왼쪽 수비수', 24),
    ('2019/2020', 'yoann court', '왼쪽 미드필더', 30),
    ('2019/2020', 'patrick burner', '오른쪽 수비수', 23),
    ('2019/2020', 'bruno ecuele manga', '중앙 수비수', 31),
    ('2019/2020', 'anthony briancon', '중앙 수비수', 25),
    ('2019/2020', 'quentin boisgard', '공격형 미드필더', 23),
    ('2020/2021', 'steven nzonzi', '중앙 수비형 미드필더', 32),
    ('2020/2021', 'ghislain konan', '왼쪽 수비수', 25),
    ('2020/2021', 'paul baysse', '중앙 수비수', 33),
    ('2020/2021', 'quentin boisgard', '공격형 미드필더', 24),
    ('2020/2021', 'andreaw gravillon', '중앙 수비수', 23),
    ('2020/2021', 'bruno ecuele manga', '중앙 수비수', 32),
    ('2020/2021', 'jordan marie', '중앙 미드필더', 29),
    ('2021/2022', 'ghislain konan', '왼쪽 수비수', 26),
    ('2021/2022', 'javairô dilrosun', '왼쪽 윙어', 24),
    ('2021/2022', 'andreaw gravillon', '중앙 수비수', 24),
    ('2021/2022', 'quentin boisgard', '공격형 미드필더', 25),
    ('2022/2023', 'muhammed cham', '공격형 미드필더', 22),
    ('2023/2024', 'muhammed cham', '공격형 미드필더', 23),
    ('2024/2025', 'anass zaroury', '왼쪽 윙어', 24),
    ('2016/2017', 'paul pogba', '중앙 미드필더', 24),
    ('2016/2017', 'erik pieters', '왼쪽 수비수', 28),
    ('2016/2017', 'jose holebas', '왼쪽 수비수', 33),
    ('2016/2017', 'ki sung-yueng', '중앙 미드필더', 28),
    ('2017/2018', 'paul pogba', '중앙 미드필더', 25),
    ('2017/2018', 'jose holebas', '왼쪽 수비수', 34),
    ('2017/2018', 'ki sung-yueng', '중앙 미드필더', 29),
    ('2017/2018', 'grzegorz krychowiak', '중앙 수비형 미드필더', 28),
    ('2017/2018', 'erik pieters', '왼쪽 수비수', 29),
    ('2018/2019', 'paul pogba', '중앙 미드필더', 26),
    ('2018/2019', 'jose holebas', '왼쪽 수비수', 35),
    ('2018/2019', 'bruno ecuele manga', '중앙 수비수', 30),
    ('2019/2020', 'erik pieters', '왼쪽 수비수', 31),
    ('2019/2020', 'teemu pukki', '중앙 공격수', 30),
    ('2020/2021', 'paul pogba', '중앙 미드필더', 28),
    ('2021/2022', 'paul pogba', '중앙 미드필더', 29),
    ('2021/2022', 'raphaël varane', '중앙 수비수', 29),
    ('2021/2022', 'wout weghorst', '중앙 공격수', 29),
    ('2021/2022', 'teemu pukki', '중앙 공격수', 32),
    ('2021/2022', 'frank onyeka', '중앙 미드필더', 24),
    ('2022/2023', 'raphaël varane', '중앙 수비수', 30),
    ('2022/2023', 'frank onyeka', '중앙 미드필더', 25),
    ('2023/2024', 'raphaël varane', '중앙 수비수', 31),
    ('2023/2024', 'frank onyeka', '중앙 미드필더', 26),
    ('2016/2017', 'josé maria callejon', '오른쪽 윙어', 30),
    ('2016/2017', 'adem ljajic', '공격형 미드필더', 25),
    ('2016/2017', 'stefan daniel radu', '중앙 수비수', 30),
    ('2016/2017', 'rade krunic', '중앙 미드필더', 23),
    ('2016/2017', 'antonino ragusa', '왼쪽 윙어', 27),
    ('2016/2017', 'ledian memushaj', '중앙 미드필더', 30),
    ('2016/2017', 'lorenzo crisetig', '중앙 수비형 미드필더', 24),
    ('2016/2017', 'nenad tomovic', '오른쪽 수비수', 29),
    ('2016/2017', 'marcello trotta', '중앙 공격수', 24),
    ('2017/2018', 'josé maria callejon', '오른쪽 윙어', 31),
    ('2017/2018', 'adem ljajic', '공격형 미드필더', 26),
    ('2017/2018', 'stefan daniel radu', '중앙 수비수', 31),
    ('2017/2018', 'luca cigarini', '중앙 수비형 미드필더', 32),
    ('2017/2018', 'marcello trotta', '중앙 공격수', 25),
    ('2017/2018', 'artur ionita', '중앙 미드필더', 27),
    ('2017/2018', 'nenad tomovic', '오른쪽 수비수', 30),
    ('2017/2018', 'alberto paloschi', '중앙 공격수', 28),
    ('2017/2018', 'antonino ragusa', '왼쪽 윙어', 28),
    ('2018/2019', 'josé maria callejon', '오른쪽 윙어', 32),
    ('2018/2019', 'steven nzonzi', '중앙 수비형 미드필더', 30),
    ('2018/2019', 'rade krunic', '중앙 미드필더', 25),
    ('2018/2019', 'luca cigarini', '중앙 수비형 미드필더', 33),
    ('2018/2019', 'stefan daniel radu', '중앙 수비수', 32),
    ('2018/2019', 'artur ionita', '중앙 미드필더', 28),
    ('2019/2020', 'josé maria callejon', '오른쪽 윙어', 33),
    ('2019/2020', 'stefan daniel radu', '중앙 수비수', 33),
    ('2019/2020', 'luca cigarini', '중앙 수비형 미드필더', 34),
    ('2019/2020', 'dalbert', '왼쪽 수비수', 26),
    ('2019/2020', 'marash kumbulla', '중앙 수비수', 20),
    ('2019/2020', 'artur ionita', '중앙 미드필더', 29),
    ('2019/2020', 'nenad tomovic', '중앙 수비수', 32),
    ('2019/2020', 'nikolas spalek', '오른쪽 미드필더', 23),
    ('2020/2021', 'stefan daniel radu', '중앙 수비수', 34),
    ('2020/2021', 'marash kumbulla', '중앙 수비수', 21),
    ('2020/2021', 'riccardo improta', '왼쪽 윙어', 28),
    ('2020/2021', 'milos vulic', '중앙 미드필더', 24),
    ('2020/2021', 'artur ionita', '중앙 미드필더', 30),
    ('2021/2022', 'rade krunic', '공격형 미드필더', 28),
    ('2021/2022', 'dalbert', '왼쪽 미드필더', 28),
    ('2021/2022', 'viktor kovalenko', '공격형 미드필더', 26),
    ('2021/2022', 'josé maria callejon', '오른쪽 윙어', 35),
    ('2022/2023', 'rade krunic', '공격형 미드필더', 29),
]

# 2. 업데이트 정보를 DataFrame으로 변환
# (순서 컬럼은 제외하고 실제 데이터만 사용)
try:
    df_truth = pd.DataFrame(update_data_list, columns=['시즌', '선수명', '포지션', '시즌종료시점만나이'])
except ValueError as e:
    print(f"❌ 제공해주신 목록의 형식이 잘못되었습니다: {e}")
    exit()

print(f"✅ 총 {len(df_truth)}건의 업데이트 정보를 리스트에서 불러왔습니다.")

# 3. 'data.csv' 파일 로드
try:
    df_main = pd.read_csv("data_corrected.csv")
    print("✅ 'data.csv' 파일을 성공적으로 불러왔습니다.")
except FileNotFoundError:
    print("❌ 'data.csv' 파일을 찾을 수 없습니다.")
    exit()
except Exception as e:
    print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
    exit()

# 4. 키(Key) 표준화: '시즌' 및 '선수명'
# '시즌' 컬럼 형식 통일 (예: '2016/2017' -> '2016-2017')
if '시즌' in df_main.columns:
    df_main['시즌'] = df_main['시즌'].astype(str).str.replace('/', '-', regex=False)
else:
    print("❌ 'data.csv'에 '시즌' 컬럼이 없습니다.")
    exit()
    
if '시즌' in df_truth.columns:
    df_truth['시즌'] = df_truth['시즌'].astype(str).str.replace('/', '-', regex=False)
else:
    print("❌ 업데이트 목록에 '시즌' 정보가 없습니다.")
    exit()

# '선수명' 공백 제거
if '선수명' in df_main.columns:
    df_main['선수명'] = df_main['선수명'].str.strip()
else:
    print("❌ 'data.csv'에 '선수명' 컬럼이 없습니다.")
    exit()

if '선수명' in df_truth.columns:
    df_truth['선수명'] = df_truth['선수명'].str.strip()
else:
    print("❌ 업데이트 목록에 '선수명' 정보가 없습니다.")
    exit()

print("✅ '시즌' 및 '선수명' 컬럼을 표준화했습니다.")

# 5. 효율적인 업데이트를 위해 인덱스 설정
# 원본 컬럼 순서 저장
original_columns = df_main.columns.tolist()

# 'data.csv'의 (선수명, 시즌) 중복 제거 (첫 번째 값 유지)
if df_main.duplicated(subset=['선수명', '시즌']).any():
    print("⚠️ 'data.csv'의 ('선수명', '시즌') 키에 중복이 발견되어, 첫 번째 값 기준으로 처리합니다.")
    df_main = df_main.drop_duplicates(subset=['선수명', '시즌'], keep='first')

df_main = df_main.set_index(['선수명', '시즌'])

# '업데이트 목록'의 (선수명, 시즌) 중복 제거 (마지막 값 유지 - 최신 정보)
if df_truth.duplicated(subset=['선수명', '시즌']).any():
    print("⚠️ '업데이트 목록'에 ('선수명', '시즌') 키에 중복이 발견되어, 마지막 값 기준으로 처리합니다.")
    df_truth = df_truth.drop_duplicates(subset=['선수명', '시즌'], keep='last')

df_truth = df_truth.set_index(['선수명', '시즌'])


# 6. 'data.csv' 업데이트 실행
# 작업 1: '포지션' 업데이트 (목록 기준으로 덮어쓰기)
print("\n[작업 1] '포지션' 컬럼을 업데이트합니다...")
original_positions = df_main['포지션'].copy()
df_main['포지션'].update(df_truth['포지션'])
position_changes = (df_main['포지션'] != original_positions).sum()
print(f"  -> 총 {position_changes}개의 '포지션' 정보가 업데이트되었습니다.")

# 작업 2: '시즌종료시점만나이'의 비어있는(NaN) 값 채우기
print("\n[작업 2] '시즌종료시점만나이'의 비어있는(NaN) 값을 업데이트 목록 기준으로 채웁니다...")
original_na_count = df_main['시즌종료시점만나이'].isna().sum()
df_main['시즌종료시점만나이'] = df_main['시즌종료시점만나이'].fillna(df_truth['시즌종료시점만나이'])
filled_na_count = original_na_count - df_main['시즌종료시점만나이'].isna().sum()
print(f"  -> 총 {filled_na_count}개의 '시즌종료시점만나이' 정보가 채워졌습니다.")


# 7. 인덱스 리셋 및 원본 컬럼 순서 복원 후 저장
df_corrected = df_main.reset_index()
df_corrected = df_corrected[original_columns] # 원본 순서 유지

output_filename = 'data_final_corrected.csv'
df_corrected.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"\n✅ 모든 작업이 완료되었습니다. '{output_filename}' 파일로 저장되었습니다.")

데이터 업데이트 스크립트를 시작합니다...
✅ 총 214건의 업데이트 정보를 리스트에서 불러왔습니다.
✅ 'data.csv' 파일을 성공적으로 불러왔습니다.
✅ '시즌' 및 '선수명' 컬럼을 표준화했습니다.
⚠️ '업데이트 목록'에 ('선수명', '시즌') 키에 중복이 발견되어, 마지막 값 기준으로 처리합니다.

[작업 1] '포지션' 컬럼을 업데이트합니다...
  -> 총 213개의 '포지션' 정보가 업데이트되었습니다.

[작업 2] '시즌종료시점만나이'의 비어있는(NaN) 값을 업데이트 목록 기준으로 채웁니다...
  -> 총 213개의 '시즌종료시점만나이' 정보가 채워졌습니다.

✅ 모든 작업이 완료되었습니다. 'data_final_corrected.csv' 파일로 저장되었습니다.


In [8]:
import pandas as pd
import numpy as np
import io

print("데이터 업데이트 스크립트를 시작합니다...")

# --- 1. 업데이트할 정보 (제공해주신 307개 목록) ---
update_data_list = [
    ('2016-2017', 'xabi alonso', '중앙 수비형 미드필더', 35),
    ('2016-2017', 'frederik sørensen', '중앙 수비수', 25),
    ('2016-2017', 'josuha guilavogui', '중앙 수비형 미드필더', 26),
    ('2016-2017', 'giulio donati', '오른쪽 수비수', 27),
    ('2016-2017', 'mark uth', '중앙 공격수', 25),
    ('2016-2017', 'maximilian philipp', '중앙 공격수', 23),
    ('2016-2017', 'levin mete öztunali', '오른쪽 미드필더', 21),
    ('2016-2017', 'simon zoller', '중앙 공격수', 25),
    ('2017-2018', 'salif sané', '중앙 수비수', 27),
    ('2017-2018', 'maximilian philipp', '중앙 공격수', 24),
    ('2017-2018', 'mark uth', '중앙 공격수', 26),
    ('2017-2018', 'josuha guilavogui', '중앙 수비형 미드필더', 27),
    ('2017-2018', 'giulio donati', '오른쪽 수비수', 28),
    ('2017-2018', 'frederik sørensen', '중앙 수비수', 26),
    ('2017-2018', 'levin mete öztunali', '오른쪽 미드필더', 22),
    ('2017-2018', 'marco terrazzino', '왼쪽 미드필더', 27),
    ('2017-2018', 'simon zoller', '중앙 공격수', 26),
    ('2017-2018', 'ishak belfodil', '중앙 공격수', 26),
    ('2018-2019', 'ishak belfodil', '중앙 공격수', 27),
    ('2018-2019', 'paco alcácer', '중앙 공격수', 25),
    ('2018-2019', 'josuha guilavogui', '중앙 수비형 미드필더', 28),
    ('2018-2019', 'jonathan schmid', '오른쪽 미드필더', 28),
    ('2018-2019', 'ondrej petrak', '중앙 수비형 미드필더', 27),
    ('2018-2019', 'mark uth', '중앙 공격수', 27),
    ('2018-2019', 'anthony ujah', '중앙 공격수', 28),
    ('2019-2020', 'mark uth', '공격형 미드필더', 28),
    ('2019-2020', 'jonathan schmid', '오른쪽 미드필더', 29),
    ('2019-2020', 'josuha guilavogui', '중앙 수비형 미드필더', 29),
    ('2019-2020', 'levin mete öztunali', '오른쪽 미드필더', 24),
    ('2019-2020', 'anthony ujah', '중앙 공격수', 29),
    ('2019-2020', 'rabbi matondo', '왼쪽 윙어', 19),
    ('2020-2021', 'maximilian philipp', '중앙 공격수', 27),
    ('2020-2021', 'jonathan schmid', '오른쪽 미드필더', 30),
    ('2020-2021', 'mark uth', '공격형 미드필더', 29),
    ('2021-2022', 'mark uth', '공격형 미드필더', 30),
    ('2021-2022', 'maximilian philipp', '중앙 공격수', 28),
    ('2021-2022', 'ishak belfodil', '중앙 공격수', 30),
    ('2021-2022', 'janni serra', '중앙 공격수', 24),
    ('2021-2022', 'bryan lasme', '중앙 공격수', 23),
    ('2022-2023', 'naouirou ahamada', '중앙 미드필더', 21),
    ('2022-2023', 'josuha guilavogui', '중앙 수비형 미드필더', 32),
    ('2022-2023', 'simon zoller', '중앙 공격수', 31),
    ('2023-2024', 'dan-axel zagadou', '중앙 수비수', 25),
    ('2016-2017', 'gabi', '중앙 미드필더', 33),
    ('2016-2017', 'vicente gomez', '중앙 수비형 미드필더', 28),
    ('2016-2017', 'alexander szymanowski', '왼쪽 미드필더', 28),
    ('2016-2017', 'samu castillejo', '오른쪽 윙어', 22),
    ('2016-2017', 'paco alcácer', '중앙 공격수', 23),
    ('2016-2017', 'carlos carmona', '중앙 미드필더', 30),
    ('2016-2017', 'jean-sylvain babin', '중앙 수비수', 30),
    ('2016-2017', 'nicola sansone', '왼쪽 윙어', 25),
    ('2016-2017', 'nabil el zhar', '오른쪽 윙어', 30),
    ('2016-2017', 'hernán pérez', '오른쪽 윙어', 28),
    ('2016-2017', 'uche henry agbo', '중앙 수비형 미드필더', 21),
    ('2016-2017', 'miguel torres', '오른쪽 수비수', 31),
    ('2016-2017', 'bruno gama', '오른쪽 미드필더', 29),
    ('2016-2017', 'gaston silva', '중앙 수비수', 23),
    ('2016-2017', 'lillo castellano', '오른쪽 수비수', 28),
    ('2016-2017', 'florin andone', '중앙 공격수', 24),
    ('2016-2017', 'miguel ángel guerrero', '중앙 공격수', 26),
    ('2017-2018', 'álex granell', '중앙 수비형 미드필더', 29),
    ('2017-2018', 'samu castillejo', '오른쪽 윙어', 23),
    ('2017-2018', 'gabi', '중앙 미드필더', 34),
    ('2017-2018', 'ryad boudebouz', '공격형 미드필더', 28),
    ('2017-2018', 'josé recio', '중앙 미드필더', 27),
    ('2017-2018', 'alejandro gálvez', '중앙 수비수', 29),
    ('2017-2018', 'miguel ángel guerrero', '중앙 공격수', 27),
    ('2017-2018', 'nabil el zhar', '오른쪽 윙어', 31),
    ('2017-2018', 'vicente gomez', '중앙 수비형 미드필더', 29),
    ('2017-2018', 'florin andone', '중앙 공격수', 25),
    ('2017-2018', 'zakaria bakkali', '왼쪽 윙어', 22),
    ('2018-2019', 'thomas lemar', '공격형 미드필더', 23),
    ('2018-2019', 'álex granell', '중앙 수비형 미드필더', 30),
    ('2018-2019', 'raúl de tomás', '중앙 공격수', 24),
    ('2018-2019', 'álex gallar', '오른쪽 윙어', 27),
    ('2018-2019', 'nabil el zhar', '오른쪽 윙어', 32),
    ('2018-2019', 'antonito', '오른쪽 수비수', 31),
    ('2019-2020', 'lumor agbenyenu', '왼쪽 수비수', 23),
    ('2019-2020', 'pablo hervias', '오른쪽 미드필더', 27),
    ('2019-2020', 'josé recio', '중앙 미드필더', 29),
    ('2019-2020', 'nikola vukcevic', '중앙 수비형 미드필더', 28),
    ('2019-2020', 'thomas lemar', '공격형 미드필더', 24),
    ('2019-2020', 'lei wu', '오른쪽 윙어', 28),
    ('2020-2021', 'thomas lemar', '공격형 미드필더', 25),
    ('2020-2021', 'josé mari', '중앙 수비형 미드필더', 33),
    ('2020-2021', 'alberto perea', '왼쪽 미드필더', 30),
    ('2020-2021', 'paco alcácer', '중앙 공격수', 27),
    ('2020-2021', 'mickaël malsa', '중앙 수비형 미드필더', 25),
    ('2020-2021', 'pablo hervias', '오른쪽 미드필더', 28),
    ('2020-2021', 'marcos mauro', '중앙 수비수', 30),
    ('2020-2021', 'salvi sánchez', '오른쪽 미드필더', 30),
    ('2020-2021', 'manuel vallejo', '중앙 공격수', 24),
    ('2020-2021', 'filip malbasic', '오른쪽 윙어', 28),
    ('2021-2022', 'raúl de tomás', '중앙 공격수', 27),
    ('2021-2022', 'thomas lemar', '공격형 미드필더', 26),
    ('2021-2022', 'alejandro gomez', '공격형 미드필더', 34),
    ('2021-2022', 'mickaël malsa', '중앙 수비형 미드필더', 26),
    ('2021-2022', 'esteban ariel saveljich', '중앙 수비수', 31),
    ('2021-2022', 'salvi sánchez', '오른쪽 미드필더', 31),
    ('2021-2022', 'javi martinez', '중앙 공격수', 22),
    ('2022-2023', 'thomas lemar', '공격형 미드필더', 27),
    ('2022-2023', 'samu castillejo', '오른쪽 윙어', 28),
    ('2023-2024', 'marc cardona', '중앙 공격수', 28),
    ('2023-2024', 'raúl de tomás', '중앙 공격수', 29),
    ('2016-2017', 'ryad boudebouz', '공격형 미드필더', 27),
    ('2016-2017', 'thomas lemar', '공격형 미드필더', 21),
    ('2016-2017', 'mario balotelli', '중앙 공격수', 26),
    ('2016-2017', 'ludovic baal', '왼쪽 수비수', 31),
    ('2016-2017', 'sèrge aurier', '오른쪽 수비수', 24),
    ('2016-2017', 'younès belhanda', '공격형 미드필더', 27),
    ('2016-2017', 'alaeddine yahia', '중앙 수비수', 35),
    ('2016-2017', 'benjamin moukandjo', '중앙 공격수', 28),
    ('2016-2017', 'arnold mvuemba', '중앙 수비형 미드필더', 32),
    ('2016-2017', 'julien cetout', '오른쪽 수비수', 29),
    ('2016-2017', 'jimmy durmaz', '중앙 미드필더', 28),
    ('2016-2017', 'modou diagne', '중앙 수비수', 23),
    ('2016-2017', 'gaël danic', '왼쪽 미드필더', 35),
    ('2016-2017', 'diallo guidileye', '중앙 수비형 미드필더', 27),
    ('2016-2017', 'youssouf hadji', '공격형 미드필더', 37),
    ('2016-2017', 'kevin berigaud', '중앙 공격수', 29),
    ('2017-2018', 'mario balotelli', '중앙 공격수', 27),
    ('2017-2018', 'thomas lemar', '공격형 미드필더', 22),
    ('2017-2018', 'dimitri liénard', '중앙 미드필더', 30),
    ('2017-2018', 'javier pastore', '공격형 미드필더', 29),
    ('2017-2018', 'thiago motta', '중앙 수비형 미드필더', 35),
    ('2017-2018', 'yoann andreu', '왼쪽 수비수', 29),
    ('2017-2018', 'bakaye dibassy', '중앙 수비수', 29),
    ('2017-2018', 'kwon chang-hoon', '공격형 미드필더', 24),
    ('2017-2018', 'jonathan rivierez', '중앙 수비수', 29),
    ('2017-2018', 'khaled adénon', '중앙 수비수', 32),
    ('2017-2018', 'edgar ié', '중앙 수비수', 24),
    ('2017-2018', 'james léa siliki', '중앙 미드필더', 22),
    ('2017-2018', 'yaya sanogo', '중앙 공격수', 25),
    ('2017-2018', 'corentin jean', '중앙 공격수', 22),
    ('2017-2018', 'anthony goncalves', '중앙 미드필더', 31),
    ('2017-2018', 'christian kouakou', '중앙 공격수', 27),
    ('2018-2019', 'sofiane alakouch', '오른쪽 수비수', 20),
    ('2018-2019', 'dimitri liénard', '중앙 미드필더', 31),
    ('2018-2019', 'anthony limbombe', '왼쪽 미드필더', 24),
    ('2018-2019', 'rachid alioui', '중앙 공격수', 26),
    ('2018-2019', 'mario balotelli', '중앙 공격수', 28),
    ('2018-2019', 'jimmy durmaz', '중앙 미드필더', 30),
    ('2018-2019', 'bakaye dibassy', '중앙 수비수', 30),
    ('2018-2019', 'lebo mothiba', '중앙 공격수', 23),
    ('2018-2019', 'khaled adénon', '중앙 수비수', 33),
    ('2018-2019', 'james léa siliki', '중앙 미드필더', 23),
    ('2018-2019', 'yaya sanogo', '중앙 공격수', 26),
    ('2018-2019', 'cheick timite', '오른쪽 윙어', 21),
    ('2019-2020', 'dimitri liénard', '중앙 미드필더', 32),
    ('2019-2020', 'ryad boudebouz', '공격형 미드필더', 30),
    ('2019-2020', 'sidy sarr', '중앙 수비형 미드필더', 24),
    ('2019-2020', 'sofiane alakouch', '오른쪽 수비수', 21),
    ('2019-2020', 'thomas delaine', '왼쪽 수비수', 28),
    ('2019-2020', 'rachid alioui', '중앙 공격수', 27),
    ('2019-2020', 'james léa siliki', '중앙 미드필더', 24),
    ('2019-2020', 'lebo mothiba', '중앙 공격수', 24),
    ('2019-2020', 'bakaye dibassy', '중앙 수비수', 31),
    ('2019-2020', 'yaya sanogo', '중앙 공격수', 27),
    ('2020-2021', 'dimitri liénard', '중앙 미드필더', 33),
    ('2020-2021', 'thomas delaine', '왼쪽 수비수', 29),
    ('2020-2021', 'ngonda muzinga', '왼쪽 수비수', 26),
    ('2020-2021', 'moreto cassamá', '중앙 미드필더', 23),
    ('2020-2021', 'lamine gueye', '왼쪽 윙어', 23),
    ('2020-2021', 'sofiane alakouch', '오른쪽 수비수', 22),
    ('2021-2022', 'dimitri liénard', '중앙 미드필더', 34),
    ('2021-2022', 'ryad boudebouz', '공격형 미드필더', 32),
    ('2021-2022', 'thomas delaine', '왼쪽 수비수', 30),
    ('2021-2022', 'jodel dossou', '오른쪽 윙어', 30),
    ('2022-2023', 'dimitri liénard', '중앙 미드필더', 35),
    ('2022-2023', 'stéphane diarra', '오른쪽 윙어', 24),
    ('2023-2024', 'mohamed daramy', '왼쪽 윙어', 22),
    ('2023-2024', 'thomas delaine', '왼쪽 수비수', 32),
    ('2023-2024', 'reda khadra', '공격형 미드필더', 23),
    ('2016-2017', 'dele alli', '공격형 미드필더', 21),
    ('2016-2017', 'nathan redmond', '왼쪽 미드필더', 23),
    ('2016-2017', 'adam clayton', '중앙 미드필더', 28),
    ('2016-2017', 'jack wilshere', '중앙 미드필더', 25),
    ('2016-2017', 'charlie adam', '중앙 미드필더', 31),
    ('2016-2017', 'joel ward', '오른쪽 수비수', 27),
    ('2016-2017', 'jack cork', '중앙 수비형 미드필더', 27),
    ('2016-2017', 'james morrison', '중앙 미드필더', 31),
    ('2016-2017', 'tom cleverley', '중앙 미드필더', 27),
    ('2016-2017', 'jordon ibe', '오른쪽 미드필더', 21),
    ('2016-2017', 'ahmed musa', '중앙 공격수', 24),
    ('2017-2018', 'dele alli', '공격형 미드필더', 22),
    ('2017-2018', 'jack wilshere', '중앙 미드필더', 26),
    ('2017-2018', 'jonjo shelvey', '중앙 미드필더', 26),
    ('2017-2018', 'joel ward', '오른쪽 수비수', 28),
    ('2017-2018', 'jack cork', '중앙 수비형 미드필더', 28),
    ('2017-2018', 'tom cleverley', '중앙 미드필더', 28),
    ('2017-2018', 'mohamed diamé', '공격형 미드필더', 30),
    ('2017-2018', 'nathan redmond', '왼쪽 미드필더', 24),
    ('2017-2018', 'tom carroll', '중앙 미드필더', 26),
    ('2017-2018', 'jordon ibe', '오른쪽 미드필더', 22),
    ('2017-2018', 'charlie austin', '중앙 공격수', 28),
    ('2018-2019', 'dele alli', '공격형 미드필더', 23),
    ('2018-2019', 'nathan redmond', '왼쪽 미드필더', 25),
    ('2018-2019', 'paul dummett', '왼쪽 수비수', 27),
    ('2018-2019', 'jack cork', '중앙 수비형 미드필더', 29),
    ('2018-2019', 'mohamed diamé', '공격형 미드필더', 31),
    ('2018-2019', 'charlie austin', '중앙 공격수', 29),
    ('2018-2019', 'florin andone', '중앙 공격수', 26),
    ('2019-2020', 'jonjo shelvey', '중앙 미드필더', 28),
    ('2019-2020', 'sèrge aurier', '오른쪽 수비수', 27),
    ('2019-2020', 'nathan redmond', '왼쪽 미드필더', 26),
    ('2019-2020', 'dele alli', '공격형 미드필더', 24),
    ('2019-2020', 'conor hourihane', '중앙 미드필더', 29),
    ('2019-2020', 'joel ward', '오른쪽 수비수', 30),
    ('2020-2021', 'patrick bamford', '중앙 공격수', 27),
    ('2020-2021', 'sèrge aurier', '오른쪽 수비수', 28),
    ('2020-2021', 'jonjo shelvey', '중앙 미드필더', 29),
    ('2020-2021', 'nathan redmond', '왼쪽 미드필더', 27),
    ('2020-2021', 'joel ward', '오른쪽 수비수', 31),
    ('2021-2022', 'jonjo shelvey', '중앙 미드필더', 30),
    ('2021-2022', 'takehiro tomiyasu', '오른쪽 수비수', 23),
    ('2021-2022', 'joel ward', '오른쪽 수비수', 32),
    ('2021-2022', 'juraj kucka', '중앙 미드필더', 35),
    ('2021-2022', 'nathan redmond', '왼쪽 미드필더', 28),
    ('2021-2022', 'jack cork', '중앙 수비형 미드필더', 32),
    ('2021-2022', 'dele alli', '공격형 미드필더', 26),
    ('2022-2023', 'sèrge aurier', '오른쪽 수비수', 30),
    ('2022-2023', 'joel ward', '오른쪽 수비수', 33),
    ('2022-2023', 'joshua dasilva', '중앙 미드필더', 24),
    ('2022-2023', 'patrick bamford', '중앙 공격수', 29),
    ('2023-2024', 'takehiro tomiyasu', '오른쪽 수비수', 25),
    ('2023-2024', 'joel ward', '오른쪽 수비수', 34),
    ('2024-2025', 'facundo buonanotte', '오른쪽 미드필더', 20),
    ('2016-2017', 'alejandro gomez', '공격형 미드필더', 29),
    ('2016-2017', 'josip ilicic', '공격형 미드필더', 29),
    ('2016-2017', 'iago falqué', '오른쪽 윙어', 27),
    ('2016-2017', 'emil hallfredsson', '중앙 미드필더', 32),
    ('2016-2017', 'thiago cionek', '중앙 수비수', 31),
    ('2016-2017', 'juraj kucka', '중앙 미드필더', 30),
    ('2016-2017', 'alessandro gazzi', '중앙 수비형 미드필더', 34),
    ('2016-2017', 'mattia destro', '중앙 공격수', 26),
    ('2016-2017', 'sinisa andelkovic', '중앙 수비수', 31),
    ('2016-2017', 'joel chukwuma obi', '중앙 미드필더', 26),
    ('2016-2017', 'federico ricci', '오른쪽 윙어', 22),
    ('2016-2017', 'roberto inglese', '중앙 공격수', 25),
    ('2016-2017', 'oscar hiljemark', '중앙 미드필더', 24),
    ('2016-2017', 'adrian stoian', '왼쪽 윙어', 26),
    ('2016-2017', 'jonathan de guzmán', '중앙 미드필더', 29),
    ('2017-2018', 'alejandro gomez', '공격형 미드필더', 30),
    ('2017-2018', 'josip ilicic', '공격형 미드필더', 30),
    ('2017-2018', 'iago falqué', '오른쪽 윙어', 28),
    ('2017-2018', 'joel chukwuma obi', '중앙 미드필더', 27),
    ('2017-2018', 'pasquale schiattarella', '중앙 수비형 미드필더', 31),
    ('2017-2018', 'roberto inglese', '중앙 공격수', 26),
    ('2017-2018', 'andrea nalini', '오른쪽 윙어', 27),
    ('2017-2018', 'mattia destro', '중앙 공격수', 27),
    ('2017-2018', 'filippo costa', '왼쪽 미드필더', 23),
    ('2017-2018', 'marco capuano', '중앙 수비수', 26),
    ('2017-2018', 'adrian stoian', '왼쪽 윙어', 27),
    ('2017-2018', 'andrey galabinov', '중앙 공격수', 29),
    ('2017-2018', 'vittorio parigini', '왼쪽 윙어', 22),
    ('2017-2018', 'maxi lopez', '중앙 공격수', 34),
    ('2018-2019', 'alejandro gomez', '공격형 미드필더', 31),
    ('2018-2019', 'josip ilicic', '공격형 미드필더', 31),
    ('2018-2019', 'iago falqué', '오른쪽 윙어', 29),
    ('2018-2019', 'pasquale schiattarella', '중앙 수비형 미드필더', 32),
    ('2018-2019', 'thiago cionek', '중앙 수비수', 33),
    ('2018-2019', 'riccardo gagliolo', '왼쪽 수비수', 29),
    ('2018-2019', 'kevin bonifazi', '중앙 수비수', 23),
    ('2018-2019', 'antonino barilla', '중앙 미드필더', 31),
    ('2018-2019', 'roberto inglese', '중앙 공격수', 27),
    ('2018-2019', 'luca siligardi', '공격형 미드필더', 31),
    ('2018-2019', 'raman chibsah', '중앙 미드필더', 26),
    ('2018-2019', 'lorenzo ariaudo', '중앙 수비수', 30),
    ('2018-2019', 'samu castillejo', '오른쪽 윙어', 24),
    ('2018-2019', 'camillo ciano', '공격형 미드필더', 29),
    ('2018-2019', 'marco capuano', '중앙 수비수', 27),
    ('2018-2019', 'fabio ceravolo', '중앙 공격수', 32),
    ('2019-2020', 'alejandro gomez', '공격형 미드필더', 32),
    ('2019-2020', 'josip ilicic', '공격형 미드필더', 32),
    ('2019-2020', 'samu castillejo', '오른쪽 윙어', 25),
    ('2019-2020', 'riccardo gagliolo', '왼쪽 수비수', 30),
    ('2019-2020', 'juraj kucka', '중앙 미드필더', 33),
    ('2019-2020', 'takehiro tomiyasu', '중앙 수비수', 21),
    ('2019-2020', 'nicola sansone', '왼쪽 윙어', 28),
    ('2019-2020', 'giulio donati', '오른쪽 수비수', 30),
    ('2019-2020', 'thiago cionek', '중앙 수비수', 34),
    ('2019-2020', 'antonino barilla', '중앙 미드필더', 32),
    ('2019-2020', 'samuel di carmine', '중앙 공격수', 31),
    ('2019-2020', 'yevhen shakhov', '중앙 미드필더', 29),
    ('2019-2020', 'alfredo donnarumma', '중앙 공격수', 29),
    ('2019-2020', 'ragnar klavan', '중앙 수비수', 34),
    ('2019-2020', 'daniele dessena', '중앙 미드필더', 33),
    ('2020-2021', 'josip ilicic', '공격형 미드필더', 33),
    ('2020-2021', 'juraj kucka', '중앙 미드필더', 34),
    ('2020-2021', 'mattia destro', '중앙 공격수', 30),
    ('2020-2021', 'pasquale schiattarella', '중앙 수비형 미드필더', 34),
    ('2020-2021', 'takehiro tomiyasu', '중앙 수비수', 22),
    ('2020-2021', 'kevin bonifazi', '중앙 수비수', 25),
    ('2020-2021', 'nicola sansone', '왼쪽 윙어', 29),
    ('2020-2021', 'riccardo gagliolo', '왼쪽 수비수', 31),
    ('2020-2021', 'samu castillejo', '오른쪽 윙어', 26),
    ('2020-2021', 'niccolo zanellato', '중앙 미드필더', 22),
    ('2020-2021', 'luca marrone', '중앙 수비수', 31),
    ('2021-2022', 'szymon zurkowski', '중앙 미드필더', 24),
    ('2021-2022', 'salva ferrer', '오른쪽 수비수', 24),
    ('2021-2022', 'nicola sansone', '왼쪽 윙어', 30),
    ('2021-2022', 'mattia destro', '중앙 공격수', 31),
    ('2021-2022', 'stefano sturaro', '중앙 미드필더', 29),
    ('2021-2022', 'tyronne ebuehi', '오른쪽 수비수', 26),
    ('2021-2022', 'joel chukwuma obi', '중앙 미드필더', 31),
    ('2022-2023', 'tyronne ebuehi', '오른쪽 수비수', 27),
    ('2023-2024', 'nicola sansone', '왼쪽 윙어', 32)
]

# 2. 업데이트 정보를 DataFrame으로 변환
try:
    df_truth = pd.DataFrame(update_data_list, columns=['시즌', '선수명', '포지션', '시즌종료시점만나이'])
except ValueError as e:
    print(f"❌ 제공해주신 목록의 형식이 잘못되었습니다: {e}")
    exit()

print(f"✅ 총 {len(df_truth)}건의 업데이트 정보를 리스트에서 불러왔습니다.")

# 3. 'data_final_corrected.csv' 파일 로드
try:
    df_main = pd.read_csv("data_final_corrected.csv")
    print("✅ 'data_final_corrected.csv' 파일을 성공적으로 불러왔습니다.")
except FileNotFoundError:
    print("❌ 'data_final_corrected.csv' 파일을 찾을 수 없습니다. 이전 단계에서 생성되었는지 확인해주세요.")
    exit()
except Exception as e:
    print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
    exit()

# 4. 키(Key) 표준화: '시즌' 및 '선수명'
# '시즌' 컬럼 형식 통일 (예: '2016/2017' -> '2016-2017')
if '시즌' in df_main.columns:
    df_main['시즌'] = df_main['시즌'].astype(str).str.replace('/', '-', regex=False)
else:
    print("❌ 'data_final_corrected.csv'에 '시즌' 컬럼이 없습니다.")
    exit()
    
if '시즌' in df_truth.columns:
    df_truth['시즌'] = df_truth['시즌'].astype(str).str.replace('/', '-', regex=False)
else:
    print("❌ 업데이트 목록에 '시즌' 정보가 없습니다.")
    exit()

# '선수명' 공백 제거
if '선수명' in df_main.columns:
    df_main['선수명'] = df_main['선수명'].str.strip()
else:
    print("❌ 'data_final_corrected.csv'에 '선수명' 컬럼이 없습니다.")
    exit()

if '선수명' in df_truth.columns:
    df_truth['선수명'] = df_truth['선수명'].str.strip()
else:
    print("❌ 업데이트 목록에 '선수명' 정보가 없습니다.")
    exit()

print("✅ '시즌' 및 '선수명' 컬럼을 표준화했습니다.")

# 5. 효율적인 업데이트를 위해 인덱스 설정
# 원본 컬럼 순서 저장
original_columns = df_main.columns.tolist()

# 'data_final_corrected.csv'의 (선수명, 시즌) 중복 제거 (첫 번째 값 유지)
if df_main.duplicated(subset=['선수명', '시즌']).any():
    print("⚠️ 'data_final_corrected.csv'의 ('선수명', '시즌') 키에 중복이 발견되어, 첫 번째 값 기준으로 처리합니다.")
    df_main = df_main.drop_duplicates(subset=['선수명', '시즌'], keep='first')

df_main = df_main.set_index(['선수명', '시즌'])

# '업데이트 목록'의 (선수명, 시즌) 중복 제거 (마지막 값 유지 - 최신 정보)
if df_truth.duplicated(subset=['선수명', '시즌']).any():
    print("⚠️ '업데이트 목록'에 ('선수명', '시즌') 키에 중복이 발견되어, 마지막 값 기준으로 처리합니다.")
    df_truth = df_truth.drop_duplicates(subset=['선수명', '시즌'], keep='last')

df_truth = df_truth.set_index(['선수명', '시즌'])


# 6. 'data_final_corrected.csv' 업데이트 실행
# 작업 1: '포지션' 업데이트 (목록 기준으로 덮어쓰기)
print("\n[작업 1] '포지션' 컬럼을 업데이트합니다...")
original_positions = df_main['포지션'].copy()
df_main['포지션'].update(df_truth['포지션'])
position_changes = (df_main['포지션'] != original_positions).sum()
print(f"  -> 총 {position_changes}개의 '포지션' 정보가 업데이트되었습니다.")

# 작업 2: '시즌종료시점만나이'의 비어있는(NaN) 값 채우기
print("\n[작업 2] '시즌종료시점만나이'의 비어있는(NaN) 값을 업데이트 목록 기준으로 채웁니다...")
original_na_count = df_main['시즌종료시점만나이'].isna().sum()
df_main['시즌종료시점만나이'] = df_main['시즌종료시점만나이'].fillna(df_truth['시즌종료시점만나이'])
filled_na_count = original_na_count - df_main['시즌종료시점만나이'].isna().sum()
print(f"  -> 총 {filled_na_count}개의 '시즌종료시점만나이' 정보가 채워졌습니다.")


# 7. 인덱스 리셋 및 원본 컬럼 순서 복원 후 저장
df_corrected = df_main.reset_index()
df_corrected = df_corrected[original_columns] # 원본 순서 유지

output_filename = 'data_final_v2_updated.csv'
df_corrected.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"\n✅ 모든 작업이 완료되었습니다. '{output_filename}' 파일로 저장되었습니다.")

데이터 업데이트 스크립트를 시작합니다...
✅ 총 307건의 업데이트 정보를 리스트에서 불러왔습니다.
✅ 'data_final_corrected.csv' 파일을 성공적으로 불러왔습니다.
✅ '시즌' 및 '선수명' 컬럼을 표준화했습니다.

[작업 1] '포지션' 컬럼을 업데이트합니다...
  -> 총 307개의 '포지션' 정보가 업데이트되었습니다.

[작업 2] '시즌종료시점만나이'의 비어있는(NaN) 값을 업데이트 목록 기준으로 채웁니다...
  -> 총 0개의 '시즌종료시점만나이' 정보가 채워졌습니다.

✅ 모든 작업이 완료되었습니다. 'data_final_v2_updated.csv' 파일로 저장되었습니다.
